# Giai đoạn 4: Kiểm định Biểu đồ, Xuất Bản & Hoàn thiện Câu chuyện (Kết luận / Hành động)

## 00. Giới thiệu giai đoạn

- **Tên giai đoạn:** Giai đoạn 4 — Hoàn thiện câu chuyện, `.ipynb` & kiểm định chất lượng trực quan (`04_visualization_and_storytelling.ipynb`).
- **Vai trò trong câu chuyện dữ liệu (*Kết luận / Hành động*):** Đóng vai trò cổng kiểm định chất lượng (QA) cuối cùng cho toàn bộ biểu đồ và nhận định từ Hồi 1, Hồi 2 và Hồi 3 trước khi xuất bộ ảnh độ phân giải cao (`dpi=300`) và hoàn thiện mạch kể chuyện *Bối cảnh → Vấn đề → Bằng chứng → Kết luận / Hành động*.
- **Mục tiêu:**
  1. Kiểm định 100% biểu đồ của Giai đoạn 1–3 theo thang nhận thức thị giác **Cleveland–McGill**, quy tắc trung thực dữ liệu (gốc 0, không 3D, không hai trục y) và **5 hướng dẫn thiết kế của Schwabish**.
  2. Xuất bộ ảnh tĩnh độ phân giải cao (`sns.set_context("talk")`, `dpi=300`, `bbox_inches="tight"`) vào thư mục đích an toàn bên ngoài `data/`/`raw_data/`.
  3. Rà soát mạch văn tiếng Việt, tính nhất quán giữa số liệu trong bảng–biểu đồ–văn bản và kiểm tra khả năng chạy tái lập từ `Kernel -> Restart & Run All` cho cả 4 notebook.
- **Phạm vi dữ liệu:** Tiếp nhận các biểu đồ, bảng và nhận định đã được thành viên xác nhận từ Notebook 01, 02, 03 (chỉ từ phân vùng `public_test`).
- **Đầu ra mong đợi (Cột mốc M4) & Nơi lưu trữ:**
  - Thư mục gốc phase: `artifacts/guided_eda/phase4_storytelling/`
  - Bảng kiểm định (`PHASE4_TABLES_DIR`): `tables/chart_audit_checklist.csv`, `tables/storytelling_final_checklist.csv`.
  - Bộ ảnh PNG nét cao (`PHASE4_FIGURES_DIR`): `figures_300dpi/` (ảnh xuất bản `dpi=300`, `context="talk"`).
- **Các giới hạn cần tuân thủ:**
  - Notebook này **không tự tạo biểu đồ hay insight thay thế** cho sản phẩm thực tế của ba notebook trước.
  - Không ghi đè hoặc xuất ảnh vào bên trong thư mục `data/` hoặc `raw_data/`.


## 01. Thiết lập môi trường


In [ ]:
import sys
from pathlib import Path

# Tự động xác định thư mục gốc dự án và thêm src vào sys.path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "common").is_dir():
    for parent in PROJECT_ROOT.parents:
        if (parent / "src" / "common").is_dir():
            PROJECT_ROOT = parent
            break

SCRIPT_ROOT = PROJECT_ROOT / "src"
if not SCRIPT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Không tìm thấy thư mục mã nguồn EDA tại: {SCRIPT_ROOT}"
    )

if str(SCRIPT_ROOT) not in sys.path:
    sys.path.insert(0, str(SCRIPT_ROOT))

print(f"Thư mục gốc dự án : {PROJECT_ROOT}")
print(f"Thư mục script EDA : {SCRIPT_ROOT}")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import có chọn lọc các tiện ích cấu hình, kiểm tra đường dẫn an toàn và xuất ảnh
from common.config import load_config
from common.paths import assert_not_in_raw, assert_safe_output, is_benchmark_file
from visualization.figure_export import export_figure

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

# Thiết lập giao diện biểu đồ cơ bản (hỗ trợ hiển thị inline trong Jupyter)
# Lưu ý: Không import visualization.figure_theme vì module đó ép backend Agg.
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titlelocation": "left",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": ":",
    "grid.alpha": 0.55,
    "axes.unicode_minus": False,
    "figure.figsize": (10, 5.5),
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

# Bảng màu gợi ý theo nguyên tắc "Bắt đầu từ màu xám" (Schwabish)
COLOR_GRAY = "#9AA5B1"
COLOR_GRAY_DARK = "#52606D"
COLOR_ACCENT_BLUE = "#1769AA"
COLOR_ACCENT_TEAL = "#087B5A"
COLOR_ACCENT_AMBER = "#A36516"
COLOR_ACCENT_RED = "#B32336"


## 02. Đọc dữ liệu, Khởi tạo thư mục đầu ra & Hợp đồng đầu vào

Notebook 04 không đọc trực tiếp dữ liệu để làm EDA lại từ đầu mà tiếp nhận các sản phẩm (biểu đồ, bảng, nhận định, giới hạn) đã được thành viên xác nhận từ Giai đoạn 1, 2 và 3, đồng thời khởi tạo (`mkdir`) các thư mục đầu ra của Giai đoạn 4.


In [ ]:
config = load_config(workspace_root=PROJECT_ROOT)
context = config.context_for("public_test")

# Khởi tạo (mkdir) thư mục lưu trữ đầu ra cho Giai đoạn 4 (nằm ngoài data/ và raw_data/)
GUIDED_ROOT = config.output_root / "guided_eda"
PHASE4_ROOT = GUIDED_ROOT / "phase4_storytelling"
PHASE4_TABLES_DIR = PHASE4_ROOT / "tables"
PHASE4_FIGURES_DIR = PHASE4_ROOT / "figures_300dpi"

assert_not_in_raw(config.raw_root, GUIDED_ROOT)
for target_dir in (PHASE4_ROOT, PHASE4_TABLES_DIR, PHASE4_FIGURES_DIR):
    assert_safe_output(config.output_root, target_dir)
    target_dir.mkdir(parents=True, exist_ok=True)

print(f"Thư mục dữ liệu gốc (chỉ đọc) : {config.raw_root}")
print(f"Thư mục artifact cho phép     : {config.output_root}")
print(f"Thư mục bảng Phase 4          : {PHASE4_TABLES_DIR}")
print(f"Thư mục ảnh xuất bản 300 dpi  : {PHASE4_FIGURES_DIR}")


In [ ]:
# ==============================================================================
# HỢP ĐỒNG ĐẦU VÀO TỪ GIAI ĐOẠN 1, 2, 3 (NB01, NB02, NB03 -> NB04)
# ==============================================================================
PHASE1_SUMMARY_DIR: Path | None = GUIDED_ROOT / "phase1_dataset_discovery"
PHASE2_SUMMARY_DIR: Path | None = GUIDED_ROOT / "phase2_business_eda"
PHASE3_SUMMARY_DIR: Path | None = GUIDED_ROOT / "phase3_cross_business"

# Danh sách các biểu đồ & bằng chứng đã được thành viên xác nhận từ Hồi 1, Hồi 2, Hồi 3
# Mỗi phần tử dự kiến là dict gồm:
# {"act": "Hồi 1|Hồi 2|Hồi 3", "task_id": "...", "chart_id": "...", "title": "...", "figure_path": "...", "limitations": "..."}
STORY_ARTIFACTS_MANIFEST: list[dict] = []


def check_phase4_input_contract(
    manifest: list[dict] = STORY_ARTIFACTS_MANIFEST,
    summary_dirs: dict[str, Path | None] | None = None,
) -> dict[str, str]:
    dirs = summary_dirs or {
        "phase1": PHASE1_SUMMARY_DIR,
        "phase2": PHASE2_SUMMARY_DIR,
        "phase3": PHASE3_SUMMARY_DIR,
    }
    report: dict[str, str] = {}
    for stage, d in dirs.items():
        if d is None:
            report[stage] = "[CHỜ BỔ SUNG] Chưa chỉ định thư mục bàn giao"
            continue
        p = Path(d)
        if not p.is_dir():
            report[stage] = f"[CHỜ BỔ SUNG] Chưa có thư mục bàn giao tại: {p}"
        else:
            csv_count = len(list(p.rglob("*.csv")))
            png_count = len(list(p.rglob("*.png")))
            if csv_count or png_count:
                report[stage] = f"[ĐÃ SẴN SÀNG] Tìm thấy {csv_count} tệp CSV và {png_count} tệp PNG trong {p.name}"
            else:
                report[stage] = f"[CHỜ BỔ SUNG] Thư mục {p.name} hiện chưa có tệp CSV hoặc PNG nào"

    if manifest:
        report["story_manifest"] = (
            f"[ĐÃ SẴN SÀNG] Đã tiếp nhận {len(manifest)} biểu đồ/hiện vật từ Hồi 1–3"
        )
    else:
        report["story_manifest"] = (
            "[CHỜ BỔ SUNG] Chờ bổ sung danh sách biểu đồ đã xác nhận từ Giai đoạn 1–3 vào STORY_ARTIFACTS_MANIFEST"
        )

    for k, v in report.items():
        print(f"[Hợp đồng đầu vào - {k}]: {v}")
    return report


phase4_contract_status = check_phase4_input_contract()


## 03. Tổng quan tiêu chuẩn kiểm định & Khung kể chuyện


In [ ]:
# Bảng tham chiếu nhanh các tiêu chuẩn trực quan hóa theo docs/DEFINE-THE-PROBLEM.md & docs/SUBJECT-CONTENT.md
standards_df = pd.DataFrame([
    {
        "Nhóm tiêu chuẩn": "Thang Cleveland–McGill",
        "Nội dung kiểm định": "Ưu tiên vị trí chung và độ dài; hạn chế diện tích/góc (tránh biểu đồ tròn > 5 lát) và màu sắc cho biến định lượng chính.",
    },
    {
        "Nhóm tiêu chuẩn": "Trung thực dữ liệu",
        "Nội dung kiểm định": "Biểu đồ cột luôn bắt đầu từ 0; không 3D; không hai trục y (dual axis); ghi rõ đơn vị, nguồn, cỡ mẫu n và khoảng ngày.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 1 — Cho thấy dữ liệu",
        "Nội dung kiểm định": "Làm nổi bật các điểm/chuỗi phục vụ lập luận chính thay vì vẽ mọi đối tượng đồng mức gây nhiễu.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 2 — Giảm nhiễu",
        "Nội dung kiểm định": "Tối đa tỷ lệ data-ink, bỏ viền trên/phải (despine), bỏ lưới đậm và nền trang trí thừa.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 3 — Gắn chữ với hình",
        "Nội dung kiểm định": "Dùng tiêu đề chủ động (active title) nêu kết luận; gắn nhãn trực tiếp (direct labeling); thêm chú thích đỉnh/đáy/ngoại lệ.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 4 — Tránh spaghetti",
        "Nội dung kiểm định": "Tách thành Small Multiples cùng thang trục và thứ tự logic khi so sánh nhiều chuỗi hoặc nhiều doanh nghiệp.",
    },
    {
        "Nhóm tiêu chuẩn": "Schwabish 5 — Bắt đầu từ màu xám",
        "Nội dung kiểm định": "Vẽ nền dữ liệu bằng màu xám trung tính; chỉ dùng màu nhấn (tối đa 5–7 màu phân loại, thân thiện mù màu) cho điểm nhấn.",
    },
    {
        "Nhóm tiêu chuẩn": "Mạch kể chuyện 3 hồi",
        "Nội dung kiểm định": "Bối cảnh (Hồi 1: 14 DN) -> Vấn đề -> Bằng chứng (Hồi 2: vi mô & Hồi 3: đối chiếu) -> Kết luận / Hành động.",
    },
])
display(standards_df)


## 04. Các tác vụ kiểm định & hoàn thiện (Tasks)


## Tác vụ T4.1a — Kiểm định toàn bộ biểu đồ Hồi 1–3 theo chuẩn Cleveland–McGill & 5 hướng dẫn Schwabish

### Mục tiêu
Rà soát, chấm điểm và phát hiện mọi lỗi trực quan hóa (sai thang đo, cắt cụt trục cột, spaghetti, thiếu tiêu đề chủ động, thiếu nhãn/đơn vị/nguồn) trên toàn bộ biểu đồ ứng viên từ Hồi 1, 2 và 3.

### Câu hỏi khám phá
- Biểu đồ nào đang dùng kênh mã hóa kém hiệu quả (góc/diện tích/màu thay vì vị trí/độ dài) hoặc vi phạm gốc 0?
- Biểu đồ nào chưa áp dụng 5 hướng dẫn Schwabish (còn nhiều màu rối, thiếu nhãn trực tiếp, tiêu đề còn thụ động)?
- Biểu đồ nào thiếu thông tin bắt buộc: đơn vị đo, cỡ mẫu `n`, nguồn dữ liệu và giới hạn phạm vi thời gian?

### Các bước thực hiện

**Bước 1 — Tập hợp danh mục biểu đồ ứng viên từ Hồi 1, 2 và 3**
Điền đầy đủ danh sách biểu đồ đã hoàn thành từ `PHASE1_SUMMARY_DIR`, `PHASE2_SUMMARY_DIR`, `PHASE3_SUMMARY_DIR` vào `STORY_ARTIFACTS_MANIFEST`.

**Bước 2 — Kiểm tra mã hóa Cleveland–McGill và nguyên tắc trung thực dữ liệu**
Kiểm tra từng biểu đồ theo các cột tiêu chí: `loại_biểu_đồ_phù_hợp`, `cột_bắt_đầu_từ_0`, `không_3D_không_2_trục_y`, `có_đơn_vị_nguồn_n`.

**Bước 3 — Chấm điểm 5 hướng dẫn của Schwabish**
Đánh giá từng biểu đồ theo 5 tiêu chí Schwabish (*Show data, Reduce clutter, Graphics+Text, Small multiples, Start with gray*), đánh dấu `ĐẠT` hoặc `CẦN SỬA`.

**Bước 4 — Kiểm tra khả năng đọc trên màn hình lớn (tương phản, cỡ chữ, mù màu)**
Rà soát độ tương phản màu nhấn trên nền xám, kiểm tra nhãn không bị chồng chéo hoặc cắt cụt.

**Bước 5 — Lập biên bản kiểm định & xuất `chart_audit_checklist.csv`**
Xuất bảng tổng hợp trạng thái kiểm định (`audit_charts_df`) vào `PHASE4_TABLES_DIR / 'chart_audit_checklist.csv'`.

### Gợi ý phương pháp & Lưu ý
- Tuyệt đối không tự bịa số liệu để vẽ lại biểu đồ trong Notebook 04 nếu chưa có dữ liệu xác nhận từ 3 notebook trước.
- Nếu phát hiện biểu đồ nào có 2 trục y (`twinx`) hoặc trục cột bị cắt khác 0, gắn trạng thái `CẦN SỬA` bắt buộc trước khi sang `T4.1b`.

### Triển khai


In [ ]:
# TODO (T4.1a — Bước 1: Tập hợp danh mục biểu đồ ứng viên từ Hồi 1, 2 và 3)
# TODO: Kiểm tra STORY_ARTIFACTS_MANIFEST và lập bảng danh sách các biểu đồ cần kiểm định


In [ ]:
# TODO (T4.1a — Bước 2: Kiểm tra mã hóa Cleveland–McGill và nguyên tắc trung thực dữ liệu)
# TODO: Xây dựng bảng checklist kiểm định Cleveland–McGill & Trung thực cho từng chart_id


In [ ]:
# TODO (T4.1a — Bước 3: Chấm điểm 5 hướng dẫn của Schwabish)
# TODO: Bổ sung cột chấm điểm 5 hướng dẫn Schwabish cho từng chart_id trong bảng kiểm định


In [ ]:
# TODO (T4.1a — Bước 4: Kiểm tra khả năng đọc trên màn hình lớn (tương phản, cỡ chữ, mù màu))
# TODO: Ghi nhận các lỗi bố cục/nhãn chữ cần tinh chỉnh trước khi xuất ảnh talk


In [ ]:
# TODO (T4.1a — Bước 5: Lập biên bản kiểm định & xuất `chart_audit_checklist.csv`)
# TODO: Tạo audit_charts_df tổng hợp kết quả kiểm định T4.1a và xuất ra PHASE4_TABLES_DIR
audit_charts_df = pd.DataFrame()


### Trực quan hóa


In [ ]:
# TODO (T4.1a — Trực quan hóa)
# Nếu có biểu đồ bị gắn cờ CẦN SỬA ở T4.1a, dùng ô này để chạy lại phiên bản đã sửa từ hàm dựng biểu đồ của nhóm và kiểm tra trực quan trước/sau.
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T4.1a):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Tổng số biểu đồ được kiểm định theo từng Hồi (Hồi 1 / Hồi 2 / Hồi 3):** ...
- **Các lỗi phổ biến phát hiện được và kết quả khắc phục:** ...


## Tác vụ T4.1b — Xuất bộ ảnh tĩnh độ phân giải cao (`sns.set_context('talk')`, `dpi=300`, `bbox_inches='tight'`)

### Mục tiêu
Thiết lập cấu hình xuất bản đồng nhất (`context='talk'`, `dpi=300`, `bbox_inches='tight'`) và xuất bộ ảnh PNG chất lượng cao vào `PHASE4_FIGURES_DIR` cho toàn bộ biểu đồ đã vượt qua cổng kiểm định `T4.1a`.

### Câu hỏi khám phá
- Khi chuyển sang `sns.set_context('talk')` (cỡ chữ lớn hơn cho trình chiếu), có nhãn hay chú thích nào bị tràn hoặc đè lên nhau không?
- Đường dẫn xuất ảnh tại `PHASE4_FIGURES_DIR` đã được kiểm tra an toàn qua `assert_not_in_raw` và `assert_safe_output` chưa?
- Tất cả tệp ảnh PNG sau khi xuất đều có kích thước hợp lệ, chữ ký PNG chuẩn và đọc rõ nét khi trình chiếu?

### Các bước thực hiện

**Bước 1 — Thiết lập cấu hình hiển thị & xuất bản `context='talk'`**
Kích hoạt `sns.set_context('talk')`, đặt `plt.rcParams['figure.dpi'] = 300`, `plt.rcParams['savefig.dpi'] = 300`, `plt.rcParams['savefig.bbox'] = 'tight'`.

**Bước 2 — Kiểm tra thư mục đích `PHASE4_FIGURES_DIR` an toàn**
Sử dụng `assert_not_in_raw(config.raw_root, PHASE4_FIGURES_DIR)` và `assert_safe_output(config.output_root, PHASE4_FIGURES_DIR)` trước khi ghi tệp.

**Bước 3 — Thực thi xuất bộ ảnh tĩnh độ phân giải cao vào `PHASE4_FIGURES_DIR`**
Gọi `export_figure(fig, output_root=config.output_root, relative_directory='guided_eda/phase4_storytelling/figures_300dpi', chart_id=..., run_id=..., include_svg=False)` hoặc `fig.savefig(PHASE4_FIGURES_DIR / f'{chart_id}.png', dpi=300, bbox_inches='tight')`.

**Bước 4 — Kiểm tra hậu kỳ chất lượng tệp ảnh sau xuất**
Xác minh sự tồn tại của từng tệp PNG trong `PHASE4_FIGURES_DIR`, dung lượng (bytes), chữ ký file PNG (`\x89PNG`) và kiểm tra bằng mắt xem chữ/nhãn có bị cắt xén không.

### Gợi ý phương pháp & Lưu ý
- Lưu ý kỹ thuật từ `visualization/figure_export.py`: hàm `export_figure` dùng `figure.dpi` và không tự truyền `bbox_inches='tight'` vào `savefig`, nên cần đặt sẵn `plt.rcParams['figure.dpi'] = 300` và `plt.rcParams['savefig.bbox'] = 'tight'` (hoặc gọi trực tiếp `fig.savefig(..., dpi=300, bbox_inches='tight')`).
- Nhớ đóng figure sau khi xuất (`plt.close(fig)`) để tránh tốn RAM khi xuất nhiều biểu đồ liên tiếp.

### Triển khai


In [ ]:
# TODO (T4.1b — Bước 1: Thiết lập cấu hình hiển thị & xuất bản `context='talk'`)
# TODO: Cấu hình thông số context='talk' và dpi=300 khi thành viên sẵn sàng xuất ảnh
# sns.set_context('talk')
# plt.rcParams.update({'figure.dpi': 300, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})


In [ ]:
# TODO (T4.1b — Bước 2: Kiểm tra thư mục đích `PHASE4_FIGURES_DIR` an toàn)
assert_not_in_raw(config.raw_root, PHASE4_FIGURES_DIR)
assert_safe_output(config.output_root, PHASE4_FIGURES_DIR)
print('Thư mục xuất ảnh an toàn:', PHASE4_FIGURES_DIR)


In [ ]:
# TODO (T4.1b — Bước 3: Thực thi xuất bộ ảnh tĩnh độ phân giải cao vào `PHASE4_FIGURES_DIR`)
# TODO: Lặp qua các biểu đồ đã đạt ở T4.1a và thực hiện xuất PNG dpi=300 vào PHASE4_FIGURES_DIR


In [ ]:
# TODO (T4.1b — Bước 4: Kiểm tra hậu kỳ chất lượng tệp ảnh sau xuất)
# TODO: Liệt kê các tệp PNG đã xuất trong PHASE4_FIGURES_DIR kèm dung lượng và trạng thái kiểm tra


### Trực quan hóa


In [ ]:
# TODO (T4.1b — Trực quan hóa)
# Hiển thị kiểm tra lại 1–2 hình ảnh PNG vừa xuất trong `PHASE4_FIGURES_DIR` bằng `IPython.display.Image` để xác nhận không bị cắt tiêu đề hoặc đè chữ ở cỡ chữ `talk`.
# Lưu ý: Không sử dụng biểu đồ 3D hoặc hai trục tung (dual axis); biểu đồ cột bắt đầu từ 0.


### Kết quả & Nhận định

> **TODO (T4.1b):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Danh sách tệp ảnh PNG `dpi=300` đã xuất tại `PHASE4_FIGURES_DIR`:** ...
- **Kết quả kiểm tra chất lượng hiển thị ở chế độ `talk`:** ...


## Tác vụ T4.1c — Hoàn thiện bộ Jupyter Notebook (`.ipynb`) tiếng Việt (Mạch văn & Tính tái lập)

### Mục tiêu
Tổng kiểm tra toàn bộ mạch kể chuyện 3 hồi (*Bối cảnh → Vấn đề → Bằng chứng → Kết luận / Hành động*), tính nhất quán 100% giữa con số trong văn bản với bảng/biểu đồ, và khả năng chạy tái lập từ đầu đến cuối của cả 4 notebook.

### Câu hỏi khám phá
- Số liệu trích dẫn trong các mục `Kết quả & Nhận định` của cả 4 notebook có khớp chính xác với con số trên biểu đồ và bảng thống kê không?
- Toàn bộ nội dung hiển thị (tiêu đề, mô tả, nhãn biểu đồ, chú thích, thuật ngữ) đã hoàn toàn bằng tiếng Việt chuẩn mực và thống nhất chưa?
- Cả 4 notebook có chạy lại mượt mà từ `Kernel -> Restart & Run All` mà không phụ thuộc thứ tự chạy ngầm hay biến toàn cục thiếu khai báo không?

### Các bước thực hiện

**Bước 1 — Rà soát mạch kể chuyện tổng thể 3 hồi**
Kiểm tra tính liên kết logic từ Hồi 1 (14 DN `NB01`) → Hồi 2 (từng DN đại diện `NB02`) → Hồi 3 (đối chiếu liên ngành `NB03`) → Kết luận & Hành động (`NB04`).

**Bước 2 — Kiểm tra tính khớp nối số liệu (Văn bản ↔ Bảng ↔ Biểu đồ)**
Đối chiếu mọi con số, tỷ lệ %, cỡ mẫu `n` và khoảng ngày được nhắc trong Markdown với kết quả tính toán thực tế.

**Bước 3 — Rà soát ngôn ngữ tiếng Việt, thuật ngữ và ghi chú giới hạn**
Đảm bảo không còn lẫn thuật ngữ tiếng Anh chưa dịch ở phần trình bày, không có diễn đạt suy diễn nhân quả hoặc quy chụp ngoài phạm vi `public_test`.

**Bước 4 — Kiểm tra khả năng tái lập (`Kernel -> Restart & Run All`)**
Chạy kiểm thử tái lập cho từng notebook (hoặc kiểm tra lệnh `jupyter nbconvert --to notebook --execute`) và ghi lại kết quả pass/fail.

**Bước 5 — Lập khung Kết luận & Hành động đề xuất và lưu `storytelling_final_checklist.csv`**
Tóm tắt các kết luận rút ra từ bằng chứng 3 hồi, xuất bảng nghiệm thu cuối cùng vào `PHASE4_TABLES_DIR / 'storytelling_final_checklist.csv'`.

### Gợi ý phương pháp & Lưu ý
- Mọi khẳng định trong mục *Kết luận / Hành động* phải gắn kèm bằng chứng cụ thể từ Hồi 1, Hồi 2 hoặc Hồi 3 — không viết nhận định chung chung không có số liệu.
- Kiểm tra lại lần cuối xem có còn cell nào chưa điền `TODO`, bị lỗi font tiếng Việt hoặc chứa đường dẫn tuyệt đối trên máy cá nhân không.

### Triển khai


In [ ]:
# TODO (T4.1c — Bước 1: Rà soát mạch kể chuyện tổng thể 3 hồi)
# TODO: Ghi nhận bản đồ mạch kể chuyện (Hồi 1 -> Hồi 2 -> Hồi 3 -> Kết luận / Hành động)


In [ ]:
# TODO (T4.1c — Bước 2: Kiểm tra tính khớp nối số liệu (Văn bản ↔ Bảng ↔ Biểu đồ))
# TODO: Viết kiểm tra đối chiếu các chỉ số chốt (headline metrics) xuyên suốt các notebook


In [ ]:
# TODO (T4.1c — Bước 3: Rà soát ngôn ngữ tiếng Việt, thuật ngữ và ghi chú giới hạn)
# TODO: Rà soát danh sách thuật ngữ và câu ghi chú giới hạn chuẩn hóa


In [ ]:
# TODO (T4.1c — Bước 4: Kiểm tra khả năng tái lập (`Kernel -> Restart & Run All`))
# TODO: Ghi nhận kết quả chạy kiểm thử tái lập (Restart & Run All) cho NB01, NB02, NB03, NB04


In [ ]:
# TODO (T4.1c — Bước 5: Lập khung Kết luận & Hành động đề xuất và lưu `storytelling_final_checklist.csv`)
# TODO: Tổng hợp bảng Kết luận & Hành động và xuất checklist cuối cùng vào PHASE4_TABLES_DIR


### Kết quả & Nhận định

> **TODO (T4.1c):** Thành viên tự ghi nhận phát hiện dựa trên bằng chứng vừa chạy ở trên (không suy diễn vượt quá dữ liệu).

- **Kết quả kiểm tra tính tái lập (`Restart & Run All`) của cả 4 notebook:** ...
- **Tóm tắt câu chuyện dữ liệu hoàn chỉnh (*Bối cảnh → Vấn đề → Bằng chứng → Kết luận / Hành động*):** ...


## 05. Tổng kết giai đoạn & Bàn giao

> **TODO (Thành viên phụ trách Giai đoạn 4):** Hoàn thiện danh mục nghiệm thu cuối cùng trước khi nộp bài.

- **Những công việc đã thực hiện:**
  - [ ] `T4.1a`: Hoàn thành kiểm định 100% biểu đồ Hồi 1–3 -> `tables/chart_audit_checklist.csv`.
  - [ ] `T4.1b`: Xuất đủ bộ ảnh PNG độ phân giải cao (`sns.set_context("talk")`, `dpi=300`, `bbox_inches="tight"`) -> `figures_300dpi/`.
  - [ ] `T4.1c`: Rà soát mạch văn tiếng Việt, đối chiếu số liệu và kiểm tra tái lập cả 4 notebook -> `tables/storytelling_final_checklist.csv`.
- **Những phát hiện & thông điệp cốt lõi toàn đồ án:**
  - ...
- **Các vấn đề còn tồn tại (nếu có):**
  - ...
- **Giới hạn tổng thể của đồ án:**
  - Chỉ phản ánh 14 doanh nghiệp trong phân vùng `public_test` trong giai đoạn 2025–2026; không suy rộng cho toàn bộ doanh nghiệp Việt Nam và không suy diễn nhân quả.
- **Hồ sơ sẵn sàng bàn giao:**
  - Bộ 4 Notebook `.ipynb` hoàn chỉnh + Bộ ảnh PNG `dpi=300` tại `artifacts/guided_eda/phase4_storytelling/figures_300dpi/` sẵn sàng đưa vào Slide thuyết trình.
